# Useful notebooks:

- Preprocessing : https://www.kaggle.com/code/motono0223/js24-preprocessing-create-lags
- Training (XGB) : https://www.kaggle.com/code/motono0223/js24-train-gbdt-model-with-lags-singlemodel
  - trained XGB model : https://www.kaggle.com/datasets/motono0223/js24-trained-gbdt-model
- Training (NN): **this notebook** https://www.kaggle.com/code/voix97/jane-street-rmf-training-nn
  - trained NN model : https://www.kaggle.com/datasets/voix97/js-xs-nn-trained-model
- Inference of NN : https://www.kaggle.com/code/voix97/jane-street-rmf-nn-with-pytorch-lightning
- Inference of NN+XGB:  https://www.kaggle.com/code/voix97/jane-street-rmf-nn-xgb
- EDA(1) : https://www.kaggle.com/code/motono0223/eda-jane-street-real-time-market-data-forecasting
- EDA(2) : https://www.kaggle.com/code/motono0223/eda-v2-jane-street-real-time-market-forecasting

# Training Neural Networks (MLP) with PyTorch Lightning

In [1]:
import os
import pickle
import polars as pl
import numpy as np
import pandas as pd

# Load Data

In [2]:
input_path = './input_df'
TRAINING = True
feature_names = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{idx}_lag_1" for idx in range(9)]
label_name = 'responder_6'
weight_name = 'weight'
train_name = os.path.join("./input_df/", "nn_input_df_with_lags.pickle")
valid_name = os.path.join("./input_df/", "nn_valid_df_with_lags.pickle")
if TRAINING and not os.path.exists(train_name):
    df = pl.scan_parquet(f"{input_path}/train.parquet").collect().to_pandas()
    valid = pl.scan_parquet(f"{input_path}/valid.parquet").collect().to_pandas()
    df = pd.concat([df, valid]).reset_index(drop=True)# A trick to boost LB from 0.0045->0.005
    with open(train_name, "wb") as w:
        pickle.dump(df, w)
    with open(valid_name, "wb") as w:
        pickle.dump(valid, w)
elif TRAINING:
    with open(train_name, "rb") as r:
        df = pickle.load(r)
    with open(valid_name, "rb") as r:
        valid = pickle.load(r)

In [ ]:
# Show the shape (rows, columns) of train and valid data
print("Train data shape:", df.shape)
print("Valid data shape:", valid.shape)

# Print the column names
print("\nTrain columns:", df.columns.tolist())
print("Valid columns:", valid.columns.tolist())

# Show the first 5 rows of train and valid data
print("\nTrain data sample:")
display(df.head())

print("\nValid data sample:")
display(valid.head())

# # Optionally, show some basic info and statistics
# print("\nTrain data info:")
# df.info()

# print("\nBasic statistics of numeric columns in train data:")
# display(df.describe())


## Select Features

In [4]:
import pandas as pd
import numpy as np

def add_rsi_features(
    df, 
    columns_to_transform, 
    date_col='date_id', 
    window=6, 
    min_periods=1
):
    """
    Adds RSI features for the given columns in `df`.
    Groups by `date_col` before computing RSI within each group.
    
    Parameters
    ----------
    df : pd.DataFrame
        The dataframe.
    columns_to_transform : list
        Column names for which RSI will be computed.
    date_col : str
        The column name that indicates 'groups' (e.g., date_id, or some grouping key).
    window : int
        The rolling window for RSI calculation. Common default is 14.
    min_periods : int
        Minimum periods for rolling calculation. 
        If set to 1, RSI starts computing with as few periods as available.
    
    Returns
    -------
    pd.DataFrame
        The original dataframe plus the new RSI columns.
    """
    def compute_rsi(series):
        # 1) Calculate daily changes
        delta = series.diff()
        
        # 2) Separate gains and losses
        gains = delta.clip(lower=0)         # keep positive changes
        losses = -1 * delta.clip(upper=0)   # keep negative changes (make positive)
        
        # 3) Calculate rolling means (simple) of gains & losses
        avg_gain = gains.rolling(window=window, min_periods=min_periods).mean()
        avg_loss = losses.rolling(window=window, min_periods=min_periods).mean()
        
        # 4) Avoid division by zero
        rs = avg_gain / (avg_loss.replace(0, np.nan))
        
        # 5) Compute RSI
        rsi = 100 - (100 / (1 + rs))
        
        return rsi
    
    for col in columns_to_transform:
        new_col_name = f"{col}_rsi{window}"
        # Group by date_col, then transform each group
        df[new_col_name] = (
            df.groupby(date_col)[col]
              .transform(lambda x: compute_rsi(x))
        )
    return df

In [5]:
# # Choose the same features you previously used for moving averages
# features_for_rsi = [f"feature_{i:02d}" for i in range(79)]

# # Add RSI features to df and valid
# df = add_rsi_features(df, features_for_rsi, date_col='date_id', window=30, min_periods=1)
# valid = add_rsi_features(valid, features_for_rsi, date_col='date_id', window=30, min_periods=1)

In [6]:
# use_rsi_features = True

# # Original feature names
# feature_names = [
#     f"feature_{i:02d}" for i in range(79)
# ] + [
#     f"responder_{idx}_lag_1" for idx in range(9)
# ]

# if use_rsi_features:
#     rsi_feature_names = [f"{col}_rsi30" for col in features_for_rsi]  
#     feature_names += rsi_feature_names

# # 5) Set label & weight
# label_name = 'responder_6'
# weight_name = 'weight'

In [7]:
# import pandas as pd
# import numpy as np
# from xgboost import XGBRegressor

# # ------------------------------------------------------------------------------
# # 1) Prepare data: Add your RSI feature columns to `feature_names`
# # ------------------------------------------------------------------------------
# # Original numeric features
# original_feature_names = [f"feature_{i:02d}" for i in range(79)]

# # Suppose you created RSI features (e.g., using a window=14)
# # that look like: "feature_00_rsi14", "feature_01_rsi14", etc.
# # We'll assume the same 79 original features got RSI columns.
# rsi_feature_names = [f"feature_{i:02d}_rsi30" for i in range(79)]

# # Combine them into a single feature list
# feature_names = original_feature_names + rsi_feature_names

# # Label & weight columns
# label_name = "responder_6"
# weight_name = "weight"

# # Extract data from the dataframe
# X = df[feature_names]
# y = df[label_name]
# w = None  # if you don't have weights, set to None

# # If you have weights:
# if weight_name in df.columns:
#     w = df[weight_name]

In [8]:
# # ------------------------------------------------------------------------------
# # 2) Train a model (XGBoost) and get overall feature importances
# # ------------------------------------------------------------------------------
# model = XGBRegressor(
#     tree_method='gpu_hist',
#     device="cuda",
#     predictor='gpu_predictor',
#     random_state=42,
#     n_estimators=100,
#     learning_rate=0.1,
#     max_depth=6,
#     objective="reg:squarederror"
# )

# print("Feature matrix shape:", X.shape, "Label shape:", y.shape)

# if w is not None:
#     model.fit(X, y, sample_weight=w)
# else:
#     model.fit(X, y)

# # Get overall feature importances
# importances = model.feature_importances_
# importance_df = pd.DataFrame({
#     "feature": feature_names,
#     "importance": importances
# })
# importance_df.sort_values("importance", ascending=False, inplace=True)
# importance_df.reset_index(drop=True, inplace=True)

In [9]:
# # ------------------------------------------------------------------------------
# # 3) Retrieve the top 20 RSI features only
# # ------------------------------------------------------------------------------
# # Filter the importance dataframe for features that contain "_rsi"
# rsi_importance_df = importance_df[importance_df["feature"].str.contains("_rsi")]

# # Sort descending and pick top 20
# rsi_importance_df.sort_values("importance", ascending=False, inplace=True)
# rsi_importance_df.reset_index(drop=True, inplace=True)

# top_20_rsi_features = rsi_importance_df.head(20)["feature"].tolist()

# print("Top 20 most important RSI features:")
# print(top_20_rsi_features)

# # ------------------------------------------------------------------------------
# # 4) (Optional) Compare with top N overall features
# # ------------------------------------------------------------------------------
# top_n = 20
# important_features_top_n = importance_df.head(top_n)["feature"].tolist()

# print("\nTop 20 overall most important features:")
# print(important_features_top_n)

## Add Features


In [10]:
# 1) Import your add_rsi_features function
#    Make sure you've defined it somewhere or imported it.
# from your_module import add_rsi_features  # <-- adjust as needed

# 2) Specify ONLY the top columns (base features) you want to compute RSI for.
#    For instance, if these columns are your original "feature_XX" columns:
top_base_features = [
    'feature_18',
    'feature_23',
    'feature_09',
    'feature_11',
    'feature_26'
]

# 3) Create RSI columns for these top features only.
#    Adjust `window=6` (or whatever window size you used to produce "_rsi6").
df = add_rsi_features(df, top_base_features, date_col='date_id', window=30, min_periods=1)
valid = add_rsi_features(valid, top_base_features, date_col='date_id', window=30, min_periods=1)

# 4) Decide if you want these new RSI features in your final model
use_rsi_features = True

# Original feature names
feature_names = [f"feature_{i:02d}" for i in range(79)] + \
                [f"responder_{idx}_lag_1" for idx in range(9)]

if use_rsi_features:
    # Construct the names of the newly generated RSI columns
    rsi_feature_names = [f"{col}_rsi30" for col in top_base_features]
    feature_names += rsi_feature_names

# 5) Set label & weight
label_name = 'responder_6'
weight_name = 'weight'

In [ ]:
X_train = df[ feature_names ]
y_train = df[ label_name ]
w_train = df[ "weight" ]
X_valid = valid[ feature_names ]
y_valid = valid[ label_name ]
w_valid = valid[ "weight" ]

X_train.shape, y_train.shape, w_train.shape, X_valid.shape, y_valid.shape, w_valid.shape

# Training Configurations

In [12]:
import os
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import (LightningDataModule, LightningModule, Trainer)
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, Timer
from pytorch_lightning.loggers import WandbLogger
import wandb
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader


class custom_args():
    def __init__(self):
        self.usegpu = True
        self.gpuid = 3
        self.seed = 42
        self.model = 'nn'
        self.use_wandb = False
        self.project = 'js-xs-nn-with-lags'
        self.dname = "./input_df/"
        self.loader_workers = 4
        self.bs = 8192
        self.lr = 1e-3
        self.weight_decay = 5e-4
        self.dropouts = [0.1, 0.1]
        self.n_hidden = [512, 512, 256]
        self.patience = 25
        self.max_epochs = 30
        self.N_fold = 5


my_args = custom_args()

# PyTorch Data Module Definition

In [13]:
class CustomDataset(Dataset):
    def __init__(self, df, accelerator):
        self.features = torch.FloatTensor(df[feature_names].values).to(accelerator)
        self.labels = torch.FloatTensor(df[label_name].values).to(accelerator)
        self.weights = torch.FloatTensor(df[weight_name].values).to(accelerator)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        w = self.weights[idx]
        return x, y, w


class DataModule(LightningDataModule):
    def __init__(self, train_df, batch_size, valid_df=None, accelerator='cpu'):
        super().__init__()
        self.df = train_df
        self.batch_size = batch_size
        self.dates = self.df['date_id'].unique()
        self.accelerator = accelerator
        self.train_dataset = None
        self.valid_df = None
        if valid_df is not None:
            self.valid_df = valid_df
        self.val_dataset = None

    def setup(self, fold=0, N_fold=5, stage=None):
        # Split dataset
        selected_dates = [date for ii, date in enumerate(self.dates) if ii % N_fold != fold]
        df_train = self.df.loc[self.df['date_id'].isin(selected_dates)]
        self.train_dataset = CustomDataset(df_train, self.accelerator)
        if self.valid_df is not None:
            df_valid = self.valid_df
            self.val_dataset = CustomDataset(df_valid, self.accelerator)

    def train_dataloader(self, n_workers=0):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=n_workers)

    def val_dataloader(self, n_workers=0):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=n_workers)
    


# NN Model Definition

In [14]:
# Custom R2 metric for validation
def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (np.average((y_true) ** 2, weights=sample_weight) + 1e-38)
    return r2


class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, dropouts, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        layers = []
        in_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.BatchNorm1d(in_dim))
            if i > 0:
                layers.append(nn.SiLU())
            if i < len(dropouts):
                layers.append(nn.Dropout(dropouts[i]))
            layers.append(nn.Linear(in_dim, hidden_dim))
            # layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1)) 
        layers.append(nn.Tanh())
        self.model = nn.Sequential(*layers)
        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []

    def forward(self, x):
        return 5 * self.model(x).squeeze(-1)  

    def training_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w  #
        loss = loss.mean()
        self.log('train_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        return loss

    def validation_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('val_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        self.validation_step_outputs.append((y_hat, y, w))
        return loss

    def on_validation_epoch_end(self):
        """Calculate validation WRMSE at the end of the epoch."""
        y = torch.cat([x[1] for x in self.validation_step_outputs]).cpu().numpy()
        if self.trainer.sanity_checking:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
        else:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
            weights = torch.cat([x[2] for x in self.validation_step_outputs]).cpu().numpy()
            # r2_val
            val_r_square = r2_val(y, prob, weights)
            self.log("val_r_square", val_r_square, prog_bar=True, on_step=False, on_epoch=True)
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5,
                                                               verbose=True)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }

    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        epoch = self.trainer.current_epoch
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

# Create PyTorch Data Module

In [ ]:

args = my_args

# checking device
device = torch.device(f'cuda:{args.gpuid}' if torch.cuda.is_available() and args.usegpu else 'cpu')
accelerator = 'gpu' if torch.cuda.is_available() and args.usegpu else 'cpu'
loader_device = 'cpu'


# Initialize Data Module

df[feature_names] = df[feature_names].fillna(method = 'ffill').fillna(0)
valid[feature_names] = valid[feature_names].fillna(method = 'ffill').fillna(0)
data_module = DataModule(df, batch_size=args.bs, valid_df=valid, accelerator=loader_device)

# Create Model and Training

In [ ]:
import gc
del df
gc.collect()
pl.seed_everything(args.seed)
for fold in range(args.N_fold):
    data_module.setup(fold, args.N_fold)
    # Obtain input dimension
    input_dim = data_module.train_dataset.features.shape[1]
    # Initialize Model
    model = NN(
        input_dim=input_dim,
        hidden_dims=args.n_hidden,
        dropouts=args.dropouts,
        lr=args.lr,
        weight_decay=args.weight_decay
    )
    # Initialize Logger
    if args.use_wandb:
        wandb_run = wandb.init(project=args.project, config=vars(args), reinit=True)
        logger = WandbLogger(experiment=wandb_run)
    else:
        logger = None
    # Initialize Callbacks
    early_stopping = EarlyStopping('val_loss', patience=args.patience, mode='min', verbose=False)
    checkpoint_callback = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1, verbose=False, filename=f"./models/nn_{fold}.model") 
    timer = Timer()
    # Initialize Trainer
    trainer = Trainer(
        max_epochs=args.max_epochs,
        accelerator=accelerator,
        devices=[args.gpuid] if args.usegpu else None,
        logger=logger,
        callbacks=[early_stopping, checkpoint_callback, timer],
        enable_progress_bar=True,
        default_root_dir="lightning_logs_3"

    )
    # Start Training
    trainer.fit(model, data_module.train_dataloader(args.loader_workers), data_module.val_dataloader(args.loader_workers))
    # You can find trained best model in your local path
    print(f'Fold-{fold} Training completed in {timer.time_elapsed("train"):.2f}s')
